# Buoy raw data processing — spectra

Read the raw MHL directional spectral data (Sydney, Byron Bay, Batemans Bay
— see `00_Buoys_Processing.ipynb` for the bulk-parameter equivalent covering
all 35 buoys) and write **one NetCDF per station** to
`inputs/Observations/processed/spectra/`.

Format and parsing logic live in
[`utils/observation/raw_spectra.py`](utils/observation/raw_spectra.py) — see
its docstring for the two raw file types (yearly peak-normalized spectra +
the paired `*SD.TXT` absolute-density scale file) and the local-time-to-UTC
conversion applied here.

**Sydney's `SYDNEYSD.TXT` is corrupted** in the raw archive (entirely null
bytes) — Sydney's output has the normalized spectral shape only, not an
absolute energy density. Byron Bay and Batemans Bay both have valid SD
files and get both.

In [1]:
from utils.config import resolve
from utils.observation import onshore_sites as obs
from utils.observation import raw_spectra as rs

## 1 · Read, join, and pivot

For each station: concatenate all yearly files, drop a benign trailing
`freq=0` artifact row present in a few files (the real 64-bin hour next to
it is complete — see the module docstring), pivot to `(time, freq)`, join
the paired SD file's `spect_den` by timestamp, and convert from local
Eastern Standard time (fixed UTC+10, no DST) to UTC.

This reads ~1.3 GB of raw text across the three stations — a minute or two.

In [2]:
meta = obs.get_coordinates()
spectra_dir = resolve("inputs/Observations/processed/spectra")

datasets = {}
for iid in rs.STATIONS:
    ds = rs.read_station_spectra(iid)
    datasets[iid] = ds
    has_density = "energy_density" in ds
    print(
        f"{iid}: {ds.sizes['time']} hours, {ds.sizes['freq']} freq bins, "
        f"energy_density={'yes' if has_density else 'NO (SD file corrupted)'}"
    )

Sydney_Buoy: 101449 hours, 64 freq bins, energy_density=NO (SD file corrupted)


ByronBay_Buoy: 77163 hours, 64 freq bins, energy_density=yes


BatemansBay_Buoy: 83733 hours, 64 freq bins, energy_density=yes


## 2 · Sanity check against the bulk data

Cross-validate: reconstruct `Hm0 = 4*sqrt(m0)` from each hour's spectrum
(`m0` = spectrum integrated over frequency) and compare against the bulk
`hs` already processed in `00_Buoys_Processing.ipynb`, for the two stations
with an absolute `energy_density`. They're different estimators (spectral
Hm0 vs. statistical Hsig) so they won't match exactly, but should track
closely — this catches join/unit/timezone mistakes, not just typos.

In [3]:
import numpy as np
import pandas as pd
import xarray as xr

for iid in ["ByronBay_Buoy", "BatemansBay_Buoy"]:
    ds = datasets[iid]
    bulk = xr.open_dataset(resolve(f"inputs/Observations/processed/{iid}.nc"))

    freq = ds.freq.values
    df_bins = np.where(freq <= 0.1 + 1e-9, 0.005, 0.01)  # raw bin width: 0.005 up to 0.1 Hz, 0.01 above
    m0 = np.nansum(ds["energy_density"].values * df_bins, axis=1)
    hm0 = 4 * np.sqrt(m0)

    bulk_hs = bulk["hs"].reindex(time=ds.time, method="nearest", tolerance=pd.Timedelta("2min"))
    ratio = hm0 / bulk_hs.values
    valid = np.isfinite(ratio) & (bulk_hs.values > 0.2)
    outliers = valid & ((ratio > 2.5) | (ratio < 0.4))

    print(f"{iid}: {valid.sum()} hourly comparisons, "
          f"{outliers.sum()} ({100 * outliers.sum() / valid.sum():.2f}%) "
          f"disagree by >2.5x — isolated raw spect_den glitches, not filtered here")

ByronBay_Buoy: 77015 hourly comparisons, 8 (0.01%) disagree by >2.5x — isolated raw spect_den glitches, not filtered here


BatemansBay_Buoy: 83620 hourly comparisons, 7 (0.01%) disagree by >2.5x — isolated raw spect_den glitches, not filtered here


## 3 · Write one NetCDF per station

In [4]:
rows = []
for iid in rs.STATIONS:
    out_path = rs.standardize_and_write(iid, meta[iid], spectra_dir)
    ds = datasets[iid]
    rows.append(
        {
            "instrument_id": iid,
            "n_hours": ds.sizes["time"],
            "n_freq": ds.sizes["freq"],
            "period_start": str(ds.time.values.min())[:10],
            "period_end": str(ds.time.values.max())[:10],
            "has_energy_density": "energy_density" in ds,
            "file": out_path.name,
        }
    )

summary = pd.DataFrame(rows).set_index("instrument_id")
print(f"wrote {len(summary)} station spectra file(s) to {spectra_dir}")
summary

wrote 3 station spectra file(s) to /home/tausiaj/GitHub-GeoOcean/HyGloW/NewSouthWales/inputs/Observations/processed/spectra


,n_hours,n_freq,period_start,period_end,has_energy_density,file
instrument_id,,,,,,
Sydney_Buoy,101449,64,1998-12-31,2011-12-31,False,Sydney_Buoy.nc
ByronBay_Buoy,77163,64,1999-10-26,2011-12-31,True,ByronBay_Buoy.nc
BatemansBay_Buoy,83733,64,2001-02-23,2011-12-31,True,BatemansBay_Buoy.nc


## 4 · Verify

In [5]:
for iid in rs.STATIONS:
    path = spectra_dir / f"{iid}.nc"
    ds = xr.open_dataset(path)
    print(f"=== {iid} ===")
    print(f"  vars  : {sorted(ds.data_vars)}")
    print(f"  dims  : {dict(ds.sizes)}")
    print(f"  period: {ds.time.values[0]} to {ds.time.values[-1]} (UTC)")
    ds.close()

=== Sydney_Buoy ===
  vars  : ['dirn', 'energy_norm', 'kurtosis', 'skewness', 'spread']
  dims  : {'time': 101449, 'freq': 64}
  period: 1998-12-31T14:00:00.000000000 to 2011-12-31T13:00:00.000000000 (UTC)
=== ByronBay_Buoy ===
  vars  : ['dirn', 'energy_density', 'energy_norm', 'kurtosis', 'skewness', 'spect_den', 'spread']
  dims  : {'time': 77163, 'freq': 64}
  period: 1999-10-26T07:00:00.000000000 to 2011-12-31T13:00:00.000000000 (UTC)
=== BatemansBay_Buoy ===
  vars  : ['dirn', 'energy_density', 'energy_norm', 'kurtosis', 'skewness', 'spect_den', 'spread']
  dims  : {'time': 83733, 'freq': 64}
  period: 2001-02-23T00:00:00.000000000 to 2011-12-31T13:00:00.000000000 (UTC)
